# Module 8: exercises  (solutions)

Randomised exercises give you a seeded generator `rng`, and the checks allow for simulation noise.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

## Exercise 1

**Standard error and confidence interval.** For the exam scores, compute `se` (standard error of the mean), then `ci_low` and `ci_high` of the **95% t-interval**
(use `stats.t.ppf(0.975, df=n-1)`).

In [ ]:
import math

import numpy as np
import pandas as pd
from scipy import stats

scores = pd.read_csv(DATA / "students.csv")["exam_score"]
n = len(scores)
se = scores.std() / math.sqrt(n)
t_star = stats.t.ppf(0.975, df=n - 1)
ci_low = scores.mean() - t_star * se
ci_high = scores.mean() + t_star * se

In [ ]:
assert abs(se - 0.743) < 0.005, f"SE = s / sqrt(n); got {se}"
assert abs(ci_low - 66.47) < 0.02 and abs(ci_high - 69.40) < 0.02, (ci_low, ci_high)
print(f"✅ Correct! 95% CI for the mean score: ({ci_low:.2f}, {ci_high:.2f})")

## Exercise 2

**Bootstrap the median.** Resample `scores` **with replacement** 5,000 times (each resample the same size as the data) and compute the median of each.
Store the 95% interval (2.5th and 97.5th percentiles of those medians) as `med_low`, `med_high`.

In [ ]:
rng = np.random.default_rng(11)
values = scores.to_numpy()
resamples = rng.choice(values, size=(5000, len(values)), replace=True)
medians = np.median(resamples, axis=1)
med_low, med_high = np.quantile(medians, [0.025, 0.975])

In [ ]:
sample_median = scores.median()
assert med_low < sample_median < med_high, "The interval should surround the sample median"
assert 2 < med_high - med_low < 6, f"Width {med_high - med_low:.2f} looks off: did you resample with replacement 5,000 times?"
print(f"✅ Correct! Median {sample_median:.1f}, 95% bootstrap CI ({med_low:.1f}, {med_high:.1f})")

## Exercise 3

**Permutation test.** Test whether mean exam score differs between students with and without a part-time job. Shuffle the pooled scores 5,000 times,
split into groups of the original sizes, and record the difference in means each time. Store the two-sided p-value as `p_perm`
(the share of shuffles whose absolute difference is at least as large as the observed absolute difference).

In [ ]:
s = pd.read_csv(DATA / "students.csv")
yes = s.loc[s["part_time_job"] == "yes", "exam_score"].to_numpy()
no = s.loc[s["part_time_job"] == "no", "exam_score"].to_numpy()
observed = no.mean() - yes.mean()

rng = np.random.default_rng(0)
pooled = np.concatenate([yes, no])
diffs = np.empty(5000)
for i in range(5000):
    rng.shuffle(pooled)
    diffs[i] = pooled[len(yes):].mean() - pooled[:len(yes)].mean()
p_perm = (np.abs(diffs) >= abs(observed)).mean()

In [ ]:
assert abs(observed - 2.9) < 0.01
assert 0.04 < p_perm < 0.10, f"p = {p_perm}. Expected about 0.07 (two-sided)"
print(f"✅ Correct! p = {p_perm:.3f}")

## Exercise 4

**Welch's t-test.** Run `stats.ttest_ind(no, yes, equal_var=False)`. Store `t_stat`, `p_value`, and `significant` (`True` if p < 0.05 else `False`).

In [ ]:
result = stats.ttest_ind(no, yes, equal_var=False)
t_stat = result.statistic
p_value = result.pvalue
significant = bool(p_value < 0.05)

In [ ]:
assert abs(t_stat - 1.81) < 0.02, f"t = {t_stat}"
assert abs(p_value - 0.0724) < 0.002
assert significant is False
print("✅ Correct! Not significant at 0.05, but that doesn't mean 'no effect'. See Exercise 8.")

## Exercise 5

**Effect size.** Write `cohens_d(a, b)` using the pooled standard deviation, then compute `d_job = cohens_d(no, yes)`.

In [ ]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_var = ((na - 1) * np.var(a, ddof=1) + (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2)
    return (np.mean(a) - np.mean(b)) / math.sqrt(pooled_var)

d_job = cohens_d(no, yes)

In [ ]:
assert abs(cohens_d([2, 3, 4], [1, 2, 3]) - 1.0) < 1e-9, "means differ by 1, pooled sd is 1"
assert abs(d_job - 0.278) < 0.005, f"d = {d_job}"
print(f"✅ Correct! d = {d_job:.2f}: a small-to-moderate effect")

## Exercise 6

**Multiple comparisons.** The loop below makes 100 tests on pure noise. Inside it, store each test's p-value (use `stats.ttest_ind(a, b).pvalue`) in `p_values`.
Then compute `n_false_positives` (p < 0.05) and `n_bonferroni` (p < 0.05/100).

In [ ]:
rng = np.random.default_rng(3)
p_values = []
for _ in range(100):
    a = rng.normal(size=20)
    b = rng.normal(size=20)
    p_values.append(stats.ttest_ind(a, b).pvalue)

n_false_positives = sum(p < 0.05 for p in p_values)
n_bonferroni = sum(p < 0.05 / 100 for p in p_values)

In [ ]:
assert len(p_values) == 100
assert n_false_positives == 5, f"got {n_false_positives}"
assert n_bonferroni == 0
print("✅ Correct! 5 of 100 pure-noise tests look 'significant'; Bonferroni removes them all.")

## Exercise 7

**Power by simulation.** If a real effect of $d=0.8$ exists (group 2 has mean 0.8, both groups have sd 1), how often does a *t*-test with **20 people per group** detect it at $\alpha=0.05$?
Run 2,000 simulated experiments and store the fraction detected as `power`.

In [ ]:
rng = np.random.default_rng(1)
hits = 0
for _ in range(2000):
    g1 = rng.normal(0.0, 1, 20)
    g2 = rng.normal(0.8, 1, 20)
    hits += stats.ttest_ind(g1, g2).pvalue < 0.05
power = hits / 2000

In [ ]:
assert 0.62 < power < 0.78, f"power = {power}. Expected about 0.69 (effect d=0.8, 20 per group)"
print(f"✅ Correct! power = {power:.0%}")

## Exercise 8

**Interpret honestly.** In the lesson, the job-vs-no-job difference had p ≈ 0.07 and a 95% CI for the difference of roughly (−0.3, 6.1) points. Which conclusion is best? Set `answer` to a letter.

* **a)** There is no effect of part-time jobs on exam scores, since p > 0.05.
* **b)** The data are compatible with both no difference and a sizeable one; the evidence is inconclusive, and a larger sample would help.
* **c)** Part-time jobs lower scores by 2.9 points, proven.
* **d)** There's a 93% chance the effect is real.

In [ ]:
answer = "b"

In [ ]:
assert answer == "b", "p > 0.05 is not evidence of no effect; and p is not the probability the effect is real"
print("✅ Correct!")